# Chinese Remainder Theorem

The Chinese Remainder Theorem (CRT) is one of the oldest proofs in number theory. It appears in Chinese mathematical texts from the 3rd century.

The question is answers is: *Given a system of congruences with coprime moduli, can we always find a solution?*

The answer is yes. And not only does a solution exist, it's unique modulo the product of the moduli.

## The Idea

Suppose you know what some unknown number $x$ satisfied:

$$x \equiv 2 \pmod{3}$$
$$x \equiv 3 \pmod{5}$$

We're not given $x$ directly, only these two remainders. Can you recover $x$?

We can do a naive search for it:

In [1]:
for x in 1:30
    if x % 3 == 2 && x % 5 == 3
        println("x = $x")
    end
end

x = 8
x = 23


We get $x = 8$ and $x = 23$ and the gap between then is $15 = 3 \times 5$.

This is what the CRT predicts. Solutions repeat with period equal to the product of the moduli.

So the complete solution is $x \equiv 8 \pmod{15}$.

## Why Coprime Moduli?

Great question.

The moduli must be coprime for a solution to be guaranteed. If they share a factor, for example:

$$x \equiv 1 \pmod{2}$$
$$x \equiv 2 \pmod{4}$$

the system becomes inconsistent where no solution exists at all.

The first condition requires $x$ to be odd. The second requires $x$ to be even.

These are mutually exclusive.

We can verify that no solution exists:

In [2]:
solutions = [x for x in 1:20 if x % 2 == 1 && x % 4 == 2]
println(isempty(solutions) ? "No solution exists" : "Solutions: $solutions")

No solution exists


## The General Statement

Given a system of $k$ congruences:

$$x \equiv a_1 \pmod{n_1}$$
$$x \equiv a_2 \pmod{n_2}$$
$$\vdots$$
$$x \equiv a_k \pmod{n_k}$$

where $n_1, n_2, \ldots, n_k$ are pairwise coprime, there exist a unique solution modulo $N = n_1 \cdot n_2 \cdots n_k$.

## Constructing the Solution

Like almost everything in cryptography, finding the solution by brute force works for small numbers but not at cryptographic scale. The CRT construction gives us a direct formula.

Let $N = n_1 \cdot n_2 \cdots n_k$. For each $i$, define:

$$N_i = \frac{N}{n_i}$$

$$M_i = N_i^{-1} \pmod{n_i}$$

Then the solution is:

$$x = \left(\sum_{i=1}^{k} a_i \cdot N_i \cdot M_i \right) \pmod{N}$$

Each term $a_i \cdot N_i \cdot M_i$ contributes exactly $a_i$ mod $n_i$ and $0$ mod all other moduli.

The sum picks up all the remainders simultaneously.

## Implementation

In [4]:
function crt(remainders::Vector{T}, moduli::Vector{T}) where T <: Integer
    @assert length(remainders) == length(moduli) "Must have same number of remainders and moduli"
    @assert all(gcd(moduli[i], moduli[j]) == 1 for i in 1:length(moduli) for j in i+1:length(moduli)) "Moduli must be pairwise coprime"

    N = prod(big.(moduli))
    x = big(0)

    for (a, n) in zip(remainders, moduli)
        Ni = N ÷ n
        Mi = invmod(Ni, n)
        x += a * Ni * Mi
    end

    return x % N
end

crt (generic function with 1 method)

We can verify:

In [5]:
remainders = [2, 3]
moduli     = [3, 5]

x = crt(remainders, moduli)
println("x = $x")
println("x mod 3 = $(x % 3)") # should be 2
println("x mod 5 = $(x % 5)") # should be 3

x = 8
x mod 3 = 2
x mod 5 = 3


## Three-Congruence Example

CRT works for any number of congruences. Let's try three:

$$x \equiv 1 \pmod{3}$$
$$x \equiv 4 \pmod{7}$$
$$x \equiv 6 \pmod{11}$$

In [6]:
remainders = [1, 4, 6]
moduli     = [3, 7, 11]

x = crt(remainders, moduli)
println("x = $x")
println("x mod 3  = $(x % 3)")   # should be 1
println("x mod 7  = $(x % 7)")   # should be 4
println("x mod 11 = $(x % 11)")  # should be 6

x = 193
x mod 3  = 1
x mod 7  = 4
x mod 11 = 6


## Why This Matters for Cryptography


CRT shows up in two important places in RSA:

1. **Håstad's broadcast attack** — if the same message is encrypted to three recipients using $e = 3$ with no padding, an attacker has three ciphertexts $c_1, c_2, c_3$ satisfying:

$$m^3 \equiv c_1 \pmod{n_1}$$
$$m^3 \equiv c_2 \pmod{n_2}$$
$$m^3 \equiv c_3 \pmod{n_3}$$

CRT recovers $m^3$ directly — and since $m < n_i$ for all $i$, taking the integer cube root gives $m$. No factoring required.

2. **RSA-CRT decryption** — a standard optimisation in real RSA implementations that uses CRT to speed up decryption by a factor of ~4.

The Håstad attack is covered in the next notebook.